In [1]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [2]:

# Define the transformation for the satellite images and masks
satellite_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])


In [3]:

# Load the pre-trained model (ensure that your model is loaded here)
model = SimpleCNN(input_size=(3, 512, 512), num_classes=5)  # Adjust num_classes as per your mask categories
model.load_state_dict(torch.load('your_trained_model.pth'))  # Load the trained weights
model.eval()  # Set the model to evaluation mode


NameError: name 'SimpleCNN' is not defined

In [ ]:

# Set up the paths to the satellite images
satellite_folder = './images/satellites'  # Folder containing satellite images
output_folder = './images/val_outputs'  # Folder to save the output images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)


In [ ]:

# Function to predict and overlay mask on satellite image
def predict_and_overlay(satellite_image_path, model, transform, output_path):
    # Load the satellite image
    satellite_image = Image.open(satellite_image_path).convert('RGB')

    # Apply transformations
    satellite_tensor = transform(satellite_image).unsqueeze(0)  # Add batch dimension

    # Predict the mask
    with torch.no_grad():
        output = model(satellite_tensor)  # Forward pass
        _, predicted_mask = torch.max(output, 1)  # Get the predicted class

    # Convert the predicted mask to an image
    predicted_mask = predicted_mask.squeeze().cpu().numpy().astype(np.uint8)

    # Overlay the predicted mask on the satellite image
    satellite_image = np.array(satellite_image)
    colored_mask = cv2.applyColorMap(predicted_mask * 50, cv2.COLORMAP_JET)  # Apply color map for visualization
    overlay = cv2.addWeighted(satellite_image, 0.7, colored_mask, 0.3, 0)

    # Save the overlay image
    result_image = Image.fromarray(overlay)
    result_image.save(output_path)


In [ ]:

# Process all images in the satellite folder
satellite_images = sorted(os.listdir(satellite_folder))

for satellite_image_name in satellite_images:
    satellite_image_path = os.path.join(satellite_folder, satellite_image_name)
    output_path = os.path.join(output_folder, f'overlay_{satellite_image_name}.png')
    
    # Predict the mask and save the overlay image
    predict_and_overlay(satellite_image_path, model, satellite_transform, output_path)

print("Predictions and overlays saved successfully!")